# TMTR — Reduced Replication
## FTS-Diffusion Paper, Section 5.3 / Fig. 6(a)

### Multi-asset, multi-protocol notebook
Supports: **S&P 500**, **GOOG**, **ZC=F** for the **TMTR** (Train on Mixture, Test on Real) experiment.

### What TMTR measures
Unlike TATR (which *grows* the training set with more synthetic data), TMTR keeps
the **total training-set length fixed** (`MIX_LENGTH = 252×5 = 5y`) and varies
the **proportion** of synthetic data inside it. The question becomes:

> "If we replace X% of real data with synthetic data, does the LSTM still
>  forecast the real test set well?"

Output curve: **MAPE vs Synthetic Proportion (%)**, sweeping over `{0, 10, …, 100}`.

### Protocols supported (vs the more numerous TATR ones)
- **`authors`**     — paper-faithful: each run regenerates `MIX_LENGTH` days from the SAME fixed `(init_state, init_segment)`. Variability across runs comes from diffusion stochasticity.
- **`split`**        — ONE long continuous trajectory (length `N_RUNS × MIX_LENGTH`) is generated up-front; each run uses a non-overlapping slice. The MC chain *evolves* across runs instead of restarting.
- **`random_init`** — each run picks a random `(init_state, init_segment)` from `SEGMENTS_INIT` (the post-init segments of the LSTM-train period). Diagnostic: separates init-effect from MC-effect.

### Persistent storage on Drive (same layout as TATR)
```
/content/drive/MyDrive/fts_diffusion/
├── architectures/{asset}/k{K}/   # Pre-trained FTS-Diffusion (shared with TATR)
├── synthetic/{asset}/k{K}/{protocol}/   # Generated synthetic series per run
├── results/tmtr/{asset}/k{K}/{protocol}/   # MAPE CSVs + summary
└── figures/tmtr/{asset}/k{K}/{protocol}/   # Final plots
```

### Paper-faithful hyperparameters
| Param | Value |
|-------|-------|
| LSTM hidden_dim | 32 |
| LSTM layers | 2 |
| Window size | 64 |
| Loss | MAE |
| Optimizer | Adam, lr=1e-2 |
| Epochs | 100 |
| Training mode | **Full-batch** |
| MIX_LENGTH | 252×5 = 1260 days (5 years) |
| Proportions | {0%, 10%, …, 100%} (11 points) |
| N_RUNS | 20 |


## 1. Environment Setup


In [ ]:
import os, sys, subprocess, time, json, shutil
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules
print(f"Running on Colab: {IS_COLAB}")

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    REPO_URL = 'https://github.com/thaidinger/ml-in-fcs.git'
    BRANCH = 'deli_work'
    CLONE_DIR = '/content/ml-in-fcs'
    if not os.path.exists(CLONE_DIR):
        subprocess.check_call(['git', 'clone', '-b', BRANCH, REPO_URL, CLONE_DIR])
    else:
        subprocess.check_call(['git', '-C', CLONE_DIR, 'pull'])

    REPO_ROOT = f'{CLONE_DIR}/fts-diffusion-ref'
    PERSIST_ROOT = '/content/drive/MyDrive/fts_diffusion'
else:
    REPO_ROOT = '/home/deli/ETH_projects/ml-in-fcs/fts-diffusion-ref'
    PERSIST_ROOT = '/home/deli/ETH_projects/ml-in-fcs/fts_diffusion_run'

# Create top-level Drive structure (subdirs created lazily once ASSET/PROTOCOL chosen)
os.makedirs(PERSIST_ROOT, exist_ok=True)
for sub in ['architectures', 'synthetic', 'results', 'figures']:
    os.makedirs(f'{PERSIST_ROOT}/{sub}', exist_ok=True)

sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
for d in ['data', 'res', 'trained_models', 'figs']:
    os.makedirs(d, exist_ok=True)

print(f"Working dir:    {os.getcwd()}")
print(f"Persistent dir: {PERSIST_ROOT}")


In [ ]:
# Install dependencies
for pkg in ['numpy', 'pandas', 'matplotlib', 'scipy', 'scikit-learn', 'torch', 'tqdm', 'tslearn', 'dtaidistance', 'yfinance']:
    try:
        __import__(pkg.replace('-', '_'))
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print("✓ Deps ready")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import mean_absolute_percentage_error as MAPE
from sklearn.preprocessing import MinMaxScaler
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

# GPU info
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print("✓ Deterministic mode ON (TF32 disabled)")

# Authors' modules
from utils.load_data import get_fts, load_actual_fts
from models.model_params import prm_params, pgm_params, pem_params
from models.train_ftsdiffusion import (
    train_ftsdiffusion,
    _has_recognition_artifacts, _has_generation_artifact, _has_evolution_artifact)
from models.sampling import generate_timeseries_ftsdiffusion
from experiments.utils_downstream import (
    get_downstream_data, init_first_segment,
    Timeseries2Dataset_Downstream, create_syn_dataset,
    concat_datasets_downstream, copy_dataset_downstream)
from experiments.predictor_lstm import LSTMPredictor, set_loss_fn

print("✓ Imports ready")


## 2. Configuration


In [ ]:
# ============================================================================
# === EXPERIMENT CONFIG ===
# Change ASSET, K, PROTOCOL to switch experiment dimension.
# All Drive paths are derived automatically.
# ============================================================================

ASSET          = 'sp500'      # 'sp500' | 'goog' | 'zcf'
K              = 14           # number of SISC clusters (paper uses 14; try 11, 12, etc.)
EXPERIMENT     = 'tmtr'       # fixed for this notebook
PROTOCOL       = 'authors'    # 'authors' | 'split' | 'random_init'

# Per-asset metadata (matches paper Appendix E.1)
ASSETS_CONFIG = {
    'sp500':    {'ticker': '^GSPC', 'start': '1980-01-01', 'end': '2020-01-01', 'pretty': 'S&P 500'},
    'sp500_us': {'ticker': '^GSPC', 'start': '1980-01-01', 'end': '2020-01-01', 'pretty': 'S&P 500 (retrained from scratch)'},
    'goog':  {'ticker': 'GOOG',  'start': '2005-01-01', 'end': '2020-01-01', 'pretty': 'GOOG'},
    'zcf':   {'ticker': 'ZC=F',  'start': '2001-01-01', 'end': '2020-01-01', 'pretty': 'ZC=F (Corn futures)'},
}
assert ASSET in ASSETS_CONFIG, f"Unknown asset {ASSET}"
assert EXPERIMENT == 'tmtr', "This notebook is for TMTR only"
assert PROTOCOL in ['authors', 'split', 'random_init'], f"Unknown protocol {PROTOCOL}"
assert isinstance(K, int) and K >= 2, f"K must be a positive integer >= 2, got {K}"

DATANAME       = ASSET
TICKER         = ASSETS_CONFIG[ASSET]['ticker']
START_DATE     = ASSETS_CONFIG[ASSET]['start']
END_DATE       = ASSETS_CONFIG[ASSET]['end']
PRETTY_NAME    = ASSETS_CONFIG[ASSET]['pretty']

# === Persistent paths derived from ASSET / K / EXPERIMENT / PROTOCOL ===
ARCH_DIR    = f'{PERSIST_ROOT}/architectures/{ASSET}/k{K}'
SYN_DIR     = f'{PERSIST_ROOT}/synthetic/{ASSET}/k{K}/{PROTOCOL}'
RES_DIR     = f'{PERSIST_ROOT}/results/{EXPERIMENT}/{ASSET}/k{K}/{PROTOCOL}'
FIG_DIR     = f'{PERSIST_ROOT}/figures/{EXPERIMENT}/{ASSET}/k{K}/{PROTOCOL}'
for d in [ARCH_DIR, SYN_DIR, RES_DIR, FIG_DIR,
          f'{ARCH_DIR}/res', f'{ARCH_DIR}/trained_models', f'{ARCH_DIR}/data']:
    os.makedirs(d, exist_ok=True)

# === Override the global prm_params for SISC's K ===
prm_params['dataname'] = ASSET
prm_params['k'] = K
pgm_params['n_patterns'] = K
pem_params['n_patterns'] = K

# === Experiment hyperparameters (paper-faithful) ===
N_RUNS         = 20                                    # paper: 20

# MIX_LENGTH is per-asset because get_downstream_data() returns only the SISC
# test split (~20% of total history). For S&P 500 (downstream ~2000d) the paper's
# 1260d (5y) fits; for GOOG/ZC=F the downstream is shorter and we need to shrink.
# Auto-shrink fallback in setup cell handles cases where even these are too big.
MIX_LENGTH_PER_ASSET = {
    'sp500':    252 * 5,    # 1260d = 5y (paper-faithful, downstream ~2000d)
    'sp500_us': 252 * 5,    # same as sp500
    'goog':     252 * 2,    #  504d = 2y (downstream only ~780d for GOOG)
    'zcf':      252 * 3,    #  756d = 3y (downstream ~1000d for ZC=F)
}
MIX_LENGTH     = MIX_LENGTH_PER_ASSET.get(ASSET, 252 * 5)

N_PROPORTIONS  = 10                                    # 11 points: 0, 10, ..., 100
PROPORTIONS    = list(range(0, 101, int(100 / N_PROPORTIONS)))   # [0, 10, ..., 100]

# === LSTM CONFIG (paper-faithful for TMTR) ===
WINDOW_SIZE    = 64
LSTM_HIDDEN    = 64                                    # paper TMTR uses h=64 (TATR uses 32)
LSTM_LAYERS    = 2
AHEAD          = 1
N_EPOCHS       = 200                                   # paper TMTR uses 200 (TATR uses 100)
LR             = 1e-2
LSTM_LOSS      = 'mse'                                 # paper TMTR uses MSE (TATR uses MAE)
DATATYPE       = 'prices'

print(f"Configuration:")
print(f"  ASSET         = {ASSET}  ({PRETTY_NAME})")
print(f"  K (clusters)  = {K}")
print(f"  EXPERIMENT    = {EXPERIMENT.upper()}")
print(f"  PROTOCOL      = {PROTOCOL!r}")
print(f"  N_RUNS        = {N_RUNS}")
print(f"  MIX_LENGTH    = {MIX_LENGTH} days ({MIX_LENGTH/252:.1f} years)  [per-asset default]")
print(f"  PROPORTIONS   = {PROPORTIONS}  ({len(PROPORTIONS)} points)")
print(f"  Total LSTMs   = {N_RUNS * len(PROPORTIONS)} (this session)")
print()
print(f"LSTM (paper TMTR):")
print(f"  hidden_dim={LSTM_HIDDEN}, layers={LSTM_LAYERS}, epochs={N_EPOCHS}, loss={LSTM_LOSS.upper()}")
print()
print(f"Drive paths (now K-aware):")
print(f"  ARCH_DIR = {ARCH_DIR}")
print(f"  SYN_DIR  = {SYN_DIR}")
print(f"  RES_DIR  = {RES_DIR}")
print(f"  FIG_DIR  = {FIG_DIR}")


## 3. Phase 1 — Train FTS-Diffusion Architecture (Once)

This step is **expensive (~1 hour on A100)** but only runs once.
**Checkpoints are shared with TATR**: if you already ran TATR for the same
`(ASSET, K)`, this step does nothing.


In [ ]:
# Sync persistent checkpoints to working dirs (so train_ftsdiffusion finds them)

def _is_for_asset(filename, asset, kind):
    if kind == 'res':
        return filename.startswith(f'sisc_{asset}_') and filename.endswith('.csv')
    if kind == 'trained_models':
        return (f'_{asset}_' in filename) and (filename.endswith('.pth') or filename.endswith('.pt'))
    if kind == 'data':
        return filename == f'{asset}_timeseries.csv'
    return False


def sync_persistent_to_working(persistent_dir, working_subdir, asset=None, kind=None):
    src = persistent_dir
    dst = os.path.join(REPO_ROOT, working_subdir)
    if not os.path.exists(src):
        return
    for f in os.listdir(src):
        src_f = os.path.join(src, f)
        dst_f = os.path.join(dst, f)
        if not os.path.isfile(src_f):
            continue
        if asset is not None and kind is not None and not _is_for_asset(f, asset, kind):
            continue
        if not os.path.exists(dst_f):
            shutil.copy(src_f, dst_f)
            print(f"  Restored {f}")


def sync_working_to_persistent(working_subdir, persistent_dir, asset=None, kind=None):
    src = os.path.join(REPO_ROOT, working_subdir)
    dst = persistent_dir
    os.makedirs(dst, exist_ok=True)
    if not os.path.exists(src):
        return
    for f in os.listdir(src):
        src_f = os.path.join(src, f)
        dst_f = os.path.join(dst, f)
        if not os.path.isfile(src_f):
            continue
        if asset is not None and kind is not None and not _is_for_asset(f, asset, kind):
            continue
        shutil.copy(src_f, dst_f)


if ASSET == 'sp500_us':
    print(f"[Note] ASSET=sp500_us: we will train FTS-Diffusion from scratch")
print(f"Checking for existing {ASSET} architecture checkpoints on Drive...")
sync_persistent_to_working(f'{ARCH_DIR}/res',             'res',             asset=ASSET, kind='res')
sync_persistent_to_working(f'{ARCH_DIR}/trained_models',  'trained_models',  asset=ASSET, kind='trained_models')
sync_persistent_to_working(f'{ARCH_DIR}/data',            'data',            asset=ASSET, kind='data')


In [ ]:
# Step 1: Download asset data if not already present
ts_path = f'data/{DATANAME}_timeseries.csv'

if ASSET == 'sp500_us':
    src_path = 'data/sp500_timeseries.csv'
    if os.path.exists(src_path) and not os.path.exists(ts_path):
        shutil.copy(src_path, ts_path)
        print(f"✓ Reused {src_path} → {ts_path}")

if not os.path.exists(ts_path):
    print(f"Downloading {TICKER} from {START_DATE} to {END_DATE}...")
    get_fts(ticker=TICKER, fts_name=DATANAME, start_date=START_DATE, end_date=END_DATE)
else:
    print(f"✓ Data already at {ts_path}")

fts = load_actual_fts(DATANAME).squeeze()
print(f"{PRETTY_NAME} series: {len(fts)} points, range [{fts.min():.4f}, {fts.max():.4f}]")


### Auto-fix: PEM device bug (apply BEFORE Phase 1 training)

The authors' `pattern_evolution_module.train_evolution_module` does not move target tensors to GPU before computing `cross_entropy`. This cell patches the source file in the cloned repo before any architecture training. **Idempotent**.


In [ ]:
# === AUTO-FIX: PEM target tensors not moved to GPU (idempotent) ===
import sys, importlib

pem_file = f'{REPO_ROOT}/models/pattern_evolution_module.py'
PATCH_MARKER = '_FTS_PATCH_DEVICE_FIX_'

with open(pem_file) as f:
    code_src = f.read()

if PATCH_MARKER in code_src:
    print("✓ PEM file already patched")
else:
    old_block = (
        "      target_pattern = batch_y[:, 0].long()\n"
        "      target_length = (batch_y[:, 1] - 10).long()\n"
        "      target_magnitude = batch_y[:, 2].float().view(-1, 1)"
    )
    new_block = (
        "      # " + PATCH_MARKER + "\n"
        "      _dev = next(model.parameters()).device\n"
        "      batch_x = batch_x.to(_dev)\n"
        "      target_pattern = batch_y[:, 0].long().to(_dev)\n"
        "      target_length = (batch_y[:, 1] - 10).long().to(_dev)\n"
        "      target_magnitude = batch_y[:, 2].float().view(-1, 1).to(_dev)"
    )
    if old_block not in code_src:
        print("⚠ Pattern not found — file may have changed.")
    else:
        code_src = code_src.replace(old_block, new_block)
        with open(pem_file, 'w') as f:
            f.write(code_src)
        print(f"✓ Patched {pem_file}")

with open(pem_file) as f:
    content = f.read()
assert PATCH_MARKER in content, "PATCH NOT APPLIED"

mods_to_purge = [m for m in list(sys.modules.keys())
                 if any(p in m for p in [
                     'pattern_evolution_module', 'pattern_recognition_module',
                     'pattern_generation_module', 'pattern_conditioned_diffusion',
                     'scaling_autoencoder', 'train_ftsdiffusion', 'load_models',
                 ])]
for m in mods_to_purge:
    del sys.modules[m]

from models.train_ftsdiffusion import (
    train_ftsdiffusion,
    _has_recognition_artifacts, _has_generation_artifact, _has_evolution_artifact)

import models.pattern_evolution_module as pem_mod
import inspect
src_code = inspect.getsource(pem_mod.train_evolution_module)
assert PATCH_MARKER in src_code, "Loaded function does NOT have patch — try Restart runtime"

print("✓ Modules reloaded; patched function is active")


In [ ]:
# Step 2: Train SISC + PGM + PEM (only if no checkpoint for this asset)
has_all = (_has_recognition_artifacts() and
           _has_generation_artifact() and
           _has_evolution_artifact())

if has_all:
    print(f"✓ All {ASSET} architecture artifacts found — skipping training.")
else:
    print(f"Training FTS-Diffusion architecture for {PRETTY_NAME} (SISC + PGM + PEM)...")
    print("Estimated time: ~50-75 min on A100")
    t0 = time.time()
    train_ftsdiffusion(fts, train_test_split=0.8, store_model=True)
    elapsed = (time.time() - t0) / 60
    print(f"✓ Architecture training complete in {elapsed:.1f} min")

print(f"Syncing {ASSET} checkpoints to Drive...")
sync_working_to_persistent('res', f'{ARCH_DIR}/res', asset=ASSET, kind='res')
sync_working_to_persistent('trained_models', f'{ARCH_DIR}/trained_models', asset=ASSET, kind='trained_models')
sync_working_to_persistent('data', f'{ARCH_DIR}/data', asset=ASSET, kind='data')
print(f"✓ {ASSET} checkpoints persisted on Drive at {ARCH_DIR}.")


## 4. Phase 2 — TMTR Loop

For each of the N_RUNS runs (× 3 protocols):
1. Generate `MIX_LENGTH = 252×5 = 1260` days of synthetic data (≈ 2-5 min on A100).
2. For each `proportion ∈ {0%, 10%, …, 100%}`:
   - Build mixture training set = real anchor × `(1-p)` ⊕ synthetic × `p`, total length `MIX_LENGTH`.
   - Train LSTM (full-batch, 200 epochs, lr=1e-2, MSE loss, hidden=64).
   - Test on real held-out test set → record MAPE.
3. Save run results to CSV after each run (idempotent — re-runs skip done proportions).


In [ ]:
# === Setup TMTR datasets ===
#
# TMTR uses a fixed-length training window (MIX_LENGTH) and a held-out test set.
# Following the paper's setup_dowmstream_tmtr() exactly:
#     real_timeseries = downstream[:MIX_LENGTH]    # first MIX_LENGTH days = the anchor pool
#     test_timeseries = downstream[MIX_LENGTH:]    # rest = held-out (LSTM eval)
#
# IMPORTANT: get_downstream_data() returns only the SISC TEST split (~20% of total
# history), not the full series. So:
#   - S&P 500 (~10k total) → downstream ~2000d → MIX_LENGTH=1260 fits (paper-faithful)
#   - GOOG    (~3.8k total) → downstream  ~780d → must shrink MIX_LENGTH
#   - ZC=F    (~5k total)  → downstream ~1000d → must shrink MIX_LENGTH
# This setup auto-shrinks MIX_LENGTH if needed and prints a clear warning.
#
# Additionally, each non-edge proportion `p` produces chunks of size
#   syn_len = MIX_LENGTH * p / 100,    real_len = MIX_LENGTH - syn_len
# and create_syn_dataset uses a sliding window of WINDOW_SIZE. So we need
# both chunks ≥ WINDOW_SIZE+1 days for unfold() to work. Otherwise that
# proportion would crash. We auto-filter PROPORTIONS to drop the unsafe ones.

def adapt_proportions(proportions, mix_length, window_size):
    """Drop proportions that would produce a chunk smaller than window_size+1 days."""
    min_chunk = window_size + 1
    keep = []
    for p in proportions:
        syn_len  = int(mix_length * p / 100)
        real_len = mix_length - syn_len
        if p == 0:
            if real_len >= min_chunk: keep.append(p)
        elif p == 100:
            if syn_len >= min_chunk: keep.append(p)
        else:
            if syn_len >= min_chunk and real_len >= min_chunk:
                keep.append(p)
    return keep


def setup_dowmstream_tmtr_adaptive(window_size, mix_length=MIX_LENGTH, proportions=None):
    """Returns:
        effective_mix_length: int (may be < mix_length if asset was too short)
        effective_proportions: list[int] (may be a subset of input proportions)
        real_ts:     1D numpy of effective_mix_length days (the real anchor pool)
        test_ts:     1D numpy of (total - effective_mix_length) days (the held-out)
        test_dataset: TensorDataset of test windows
        scaler:      MinMaxScaler fitted on test_ts
        init_state, init_segment: starting (state, segment) for FTS-Diffusion sampling
        segments_init / labels_init / lengths_init: SISC segments inside real_ts (for random_init)
    """
    if proportions is None:
        proportions = PROPORTIONS

    downstream_ts, segments_test, labels_test, lengths_test = get_downstream_data()
    total_len = len(downstream_ts)

    # Need ≥ window_size+1 days for ≥ 1 LSTM test window; want ≥ 252d (1y) for meaningful eval.
    min_test = max(window_size + 1, 252)

    if total_len < mix_length + min_test:
        # Auto-shrink: keep ≥ 40% of the downstream as test set
        new_mix_length = max(int(total_len * 0.6), 252)
        if new_mix_length + min_test > total_len:
            raise ValueError(
                f"Asset {ASSET} too short for TMTR even after auto-shrink: "
                f"downstream={total_len}d, need ≥ {252 + min_test}d. "
                f"Try a different asset or rerun SISC with a larger train_test_split.")
        print(f"⚠ Asset {ASSET} downstream is only {total_len}d.")
        print(f"  Auto-shrinking MIX_LENGTH: {mix_length} → {new_mix_length} days "
              f"(keeping ≥ {min_test}d as test set).")
        mix_length = new_mix_length

    # Filter PROPORTIONS to those that yield chunks ≥ window_size+1 on both sides
    eff_proportions = adapt_proportions(proportions, mix_length, window_size)
    if len(eff_proportions) < len(proportions):
        dropped = sorted(set(proportions) - set(eff_proportions))
        print(f"⚠ Dropping {len(dropped)} proportion(s) too small for WINDOW_SIZE={window_size}: {dropped}")
        print(f"  (chunk would be < {window_size + 1} days — sliding window can't fit)")
        print(f"  Effective sweep: {eff_proportions}")

    real_ts = downstream_ts[:mix_length]
    test_ts = downstream_ts[mix_length:]

    _, scaler = Timeseries2Dataset_Downstream(test_ts, window_size)
    test_dataset = Timeseries2Dataset_Downstream(test_ts, window_size, scaler)

    init_state, init_segment = init_first_segment(segments_test, labels_test, lengths_test)

    # Find which SISC segments fall inside real_ts (used by the random_init protocol)
    cum = 0
    boundary_idx = 0
    for i, l in enumerate(lengths_test):
        if cum + l <= mix_length:
            cum += l
            boundary_idx = i + 1
        else:
            break
    segments_init = segments_test[:boundary_idx]
    labels_init   = labels_test[:boundary_idx]
    lengths_init  = lengths_test[:boundary_idx]

    print(f"TMTR setup for {ASSET} ({PRETTY_NAME}):")
    print(f"  Total downstream timeseries: {total_len} days ({total_len/252:.2f} years)")
    print(f"  MIX_LENGTH (real anchor):    {mix_length} days ({mix_length/252:.2f} years)")
    print(f"  Test set (held out):         {len(test_ts)} days ({len(test_ts)/252:.2f} years)")
    print(f"  test_dataset shape:          {test_dataset.shape}")
    print(f"  segments_init: {len(segments_init)} SISC segments inside the real anchor")
    print(f"  PROPORTIONS sweep:           {eff_proportions}  ({len(eff_proportions)} points)")
    return (mix_length, eff_proportions, real_ts, test_ts, test_dataset, scaler,
            init_state, init_segment,
            segments_init, labels_init, lengths_init)


(MIX_LENGTH, PROPORTIONS, real_ts, test_ts, test_dataset, scaler,
 init_state, init_segment,
 SEGMENTS_INIT, LABELS_INIT, LENGTHS_INIT) = setup_dowmstream_tmtr_adaptive(
    WINDOW_SIZE, MIX_LENGTH, PROPORTIONS)
# ^ MIX_LENGTH and PROPORTIONS are rebound here so downstream cells
#   (run_single_tmtr, _ensure_split_trajectory, MAIN LOOP, aggregation)
#   use the effective values.


In [ ]:
# === LSTM training & evaluation functions (paper-faithful TMTR setup) ===

def train_lstm_full_batch(dataset, n_epochs=N_EPOCHS, hidden_dim=LSTM_HIDDEN,
                          n_layers=LSTM_LAYERS, output_dim=AHEAD, criterion_str=LSTM_LOSS,
                          lr=LR, seed=0):
    """Match authors' full-batch training in predictor_lstm.py."""
    torch.manual_seed(seed)
    model = LSTMPredictor(input_dim=1, hidden_dim=hidden_dim, output_dim=output_dim,
                          n_layers=n_layers, device=device).to(device)
    criterion = set_loss_fn(criterion_str)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    X = dataset[:, :-output_dim].unsqueeze(-1).to(device)
    y = dataset[:, -output_dim:].to(device)

    for epoch in range(n_epochs):
        model.train()
        y_pred = model(X)
        loss = criterion(y_pred, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    return model, float(loss.item())


def evaluate_lstm(model, test_dataset, scaler, output_dim=AHEAD):
    """Compute MAPE on the real test set (un-scaled)."""
    model.eval()
    with torch.no_grad():
        X = test_dataset[:, :-output_dim].unsqueeze(-1).to(device)
        y_true = test_dataset[:, -output_dim:].numpy()
        y_pred = model(X).cpu().numpy()
        y_true_orig = scaler.inverse_transform(y_true)
        y_pred_orig = scaler.inverse_transform(y_pred)
    return float(MAPE(y_true_orig, y_pred_orig))


def build_mix_dataset(real_ts, syn_ts, real_len, syn_len, scaler):
    """Build the mixture training dataset of total length MIX_LENGTH.

    Mirrors authors' create_mixture_dataset: take the first `real_len` days of real
    and the first `syn_len` days of synthetic, concatenate as TensorDatasets.

    Edge cases:
      - syn_len == 0:                 all-real (real-only baseline)
      - real_len == 0:                all-synthetic
      - one side < WINDOW_SIZE+1:     too short to unfold any LSTM window; we
        fall back to using only the longer side (the proportion effectively
        snaps to 0% or 100% near the boundary). This kicks in for short
        MIX_LENGTH on small assets (e.g. GOOG with MIX_LENGTH=504, prop=10%
        gives syn_len=50 < 64=WINDOW_SIZE).
    """
    real_ok = real_len >= WINDOW_SIZE + 1
    syn_ok  = syn_len  >= WINDOW_SIZE + 1
    if not real_ok and not syn_ok:
        # Should be impossible: MIX_LENGTH >= WINDOW_SIZE+1 must hold.
        raise ValueError(
            f"Both chunks too short for window={WINDOW_SIZE}: "
            f"real_len={real_len}, syn_len={syn_len}, MIX_LENGTH={MIX_LENGTH}")
    if syn_len == 0 or not syn_ok:
        if not syn_ok and syn_len > 0:
            print(f"    [build_mix] syn chunk {syn_len}d < window {WINDOW_SIZE}+1 - using real-only")
        return create_syn_dataset(real_ts[:real_len], WINDOW_SIZE, scaler, DATATYPE)
    if real_len == 0 or not real_ok:
        if not real_ok and real_len > 0:
            print(f"    [build_mix] real chunk {real_len}d < window {WINDOW_SIZE}+1 - using synth-only")
        return create_syn_dataset(syn_ts[:syn_len], WINDOW_SIZE, scaler, DATATYPE)
    # Both sides large enough -> standard mix
    real_chunk = real_ts[:real_len]
    syn_chunk  = syn_ts[:syn_len]
    real_ds = create_syn_dataset(real_chunk, WINDOW_SIZE, scaler, DATATYPE)
    syn_ds  = create_syn_dataset(syn_chunk,  WINDOW_SIZE, scaler, DATATYPE)
    return concat_datasets_downstream(real_ds, syn_ds)

print("✓ Train/eval/mix functions ready")


In [ ]:
# === TMTR run for a single seed (3 protocols: authors / split / random_init) ===

@torch.no_grad()
def generate_with_states(T, init_state, init_segment):
    """Generate a continuous synthetic trajectory of length T and return (timeseries, states)."""
    from models.load_models import load_ftsdiffusion
    from models.utils_sampling import sampling_inputs
    from models.sampling import state_evolution_ftsdiffusion, segment_generation_ftsdiffusion

    model = load_ftsdiffusion()
    l_min = prm_params['l_min']
    _, _, patterns = sampling_inputs()
    dev = next(model['evolution'].parameters()).device
    patterns = torch.tensor(patterns, dtype=torch.float32).to(dev)

    timeseries = list(init_segment)
    if hasattr(init_state, 'to'):
        state = init_state.to(dev)
    else:
        state = torch.tensor(init_state, dtype=torch.float32).to(dev).view(1, -1)

    states_list = [state.squeeze(0).cpu().numpy().astype(np.float32).copy()]
    curr_T = len(timeseries)

    while curr_T < T:
        state = state_evolution_ftsdiffusion(model['evolution'], state, l_min)
        segment = segment_generation_ftsdiffusion(model['generation'], state, patterns)
        segment = segment - segment[0] + timeseries[-1]
        seg_len = len(segment)
        timeseries.extend(segment)
        curr_T += seg_len
        states_list.append(state.squeeze(0).cpu().numpy().astype(np.float32).copy())

    timeseries_arr = np.array(timeseries[:T], dtype=np.float32)
    states_arr = np.stack(states_list, axis=0)
    return timeseries_arr, states_arr


def _ensure_split_trajectory(n_runs, mix_length, seed_base=42):
    """For 'split' protocol: generate ONE long continuous trajectory of length n_runs*mix_length.

    Cached at SYN_DIR/_global_continuous.npy (shared across runs of this protocol).
    Returns the full numpy array; run r reads slice [r*mix_length : (r+1)*mix_length].
    """
    cont_path   = f'{SYN_DIR}/_global_continuous.npy'
    states_path = f'{SYN_DIR}/_global_states.npy'
    needed = n_runs * mix_length

    if os.path.exists(cont_path):
        existing = np.load(cont_path)
        if len(existing) >= needed:
            print(f"  [split] Reusing cached global trajectory ({len(existing)} days, need {needed})")
            return existing[:needed]

    print(f"  [split] Generating ONE continuous trajectory of {needed} days "
          f"({n_runs} runs × {mix_length} days = {needed/252:.1f} years)...")
    t0 = time.time()
    torch.manual_seed(seed_base)   # fixed seed for the shared chain
    np.random.seed(seed_base)
    full_syn, states_arr = generate_with_states(needed, init_state, init_segment)
    np.save(cont_path, full_syn)
    np.save(states_path, states_arr)
    print(f"  [split] Generation: {(time.time()-t0)/60:.1f} min  (states: {states_arr.shape})")
    return full_syn[:needed]


def run_single_tmtr(run_idx, proportions, seed_base=42, protocol=None,
                    split_trajectory=None):
    """Run one TMTR replicate with the chosen protocol.

    Protocols:
      'authors':     fresh MIX_LENGTH-day synthetic series, fixed (init_state, init_segment)
      'split':       slice run_idx of the shared continuous trajectory
      'random_init': fresh MIX_LENGTH-day synthetic series, init_segment sampled from SEGMENTS_INIT
    """
    if protocol is None:
        protocol = PROTOCOL

    run_csv = f'{RES_DIR}/run_{run_idx:02d}.csv'
    results = {}
    if os.path.exists(run_csv):
        df_prev = pd.read_csv(run_csv)
        results = dict(zip(df_prev['proportion_pct'].astype(int), df_prev['mape'].astype(float)))
        if set(proportions).issubset(set(results.keys())):
            print(f"  ✓ Run {run_idx}: all proportions already in CSV — skipping.")
            return {p: results[p] for p in proportions}
        elif results:
            done = sorted(results.keys())
            missing = sorted(set(proportions) - set(done))
            print(f"  ⚠ Run {run_idx} partial: done={done}, missing={missing}")

    np.random.seed(seed_base + run_idx)
    torch.manual_seed(seed_base + run_idx)

    # ====================================================================
    # PROTOCOL: 'authors' — paper-faithful, fixed init each run
    # ====================================================================
    if protocol == 'authors':
        syn_path = f'{SYN_DIR}/run_{run_idx:02d}_syn.npy'
        if os.path.exists(syn_path):
            syn_ts = np.load(syn_path)
            if len(syn_ts) >= MIX_LENGTH:
                print(f"  Loading cached synthetic series for run {run_idx} (protocol=authors)")
                syn_ts = syn_ts[:MIX_LENGTH]
            else:
                syn_ts = None
        else:
            syn_ts = None

        if syn_ts is None:
            print(f"  Generating fresh {MIX_LENGTH}-day synthetic series for run {run_idx}...")
            t0 = time.time()
            syn_ts = generate_timeseries_ftsdiffusion(
                T=MIX_LENGTH, init_state=init_state, init_segment=init_segment)
            syn_ts = np.asarray(syn_ts, dtype=np.float32)[:MIX_LENGTH]
            if len(syn_ts) < MIX_LENGTH:
                syn_ts = np.concatenate([
                    syn_ts,
                    np.full(MIX_LENGTH - len(syn_ts), syn_ts[-1] if len(syn_ts) > 0 else 0.0)
                ])
            np.save(syn_path, syn_ts)
            print(f"  Generation: {(time.time()-t0)/60:.1f} min")

    # ====================================================================
    # PROTOCOL: 'split' — shared continuous chain, sliced by run index
    # ====================================================================
    elif protocol == 'split':
        assert split_trajectory is not None, "split protocol requires split_trajectory"
        start = run_idx * MIX_LENGTH
        end   = start + MIX_LENGTH
        syn_ts = split_trajectory[start:end].astype(np.float32)
        # Cache per-run slice for reproducibility
        np.save(f'{SYN_DIR}/run_{run_idx:02d}_slice.npy', syn_ts)
        print(f"  [split] Run {run_idx} uses slice [{start}:{end}] of shared chain")

    # ====================================================================
    # PROTOCOL: 'random_init' — random init_segment per run
    # ====================================================================
    elif protocol == 'random_init':
        syn_path = f'{SYN_DIR}/run_{run_idx:02d}_syn.npy'
        if os.path.exists(syn_path):
            syn_ts = np.load(syn_path)
            if len(syn_ts) >= MIX_LENGTH:
                print(f"  Loading cached synthetic series for run {run_idx} (protocol=random_init)")
                syn_ts = syn_ts[:MIX_LENGTH]
            else:
                syn_ts = None
        else:
            syn_ts = None

        if syn_ts is None:
            rng_local = np.random.RandomState(seed_base + run_idx + 9999)
            rand_idx = rng_local.randint(0, len(SEGMENTS_INIT))
            rand_init_segment = SEGMENTS_INIT[rand_idx]
            rand_init_pattern = LABELS_INIT[rand_idx]
            rand_init_length  = LENGTHS_INIT[rand_idx]
            rand_init_magnitude = max(rand_init_segment) - min(rand_init_segment)
            rand_init_state = np.stack(
                (rand_init_pattern, rand_init_length, rand_init_magnitude), axis=0)
            rand_init_state = torch.tensor(rand_init_state).view(1, -1)

            print(f"  Generating fresh synthetic for run {run_idx} (random_init: seg_idx={rand_idx})...")
            t0 = time.time()
            syn_ts = generate_timeseries_ftsdiffusion(
                T=MIX_LENGTH, init_state=rand_init_state, init_segment=rand_init_segment)
            syn_ts = np.asarray(syn_ts, dtype=np.float32)[:MIX_LENGTH]
            if len(syn_ts) < MIX_LENGTH:
                syn_ts = np.concatenate([
                    syn_ts,
                    np.full(MIX_LENGTH - len(syn_ts), syn_ts[-1] if len(syn_ts) > 0 else 0.0)
                ])
            np.save(syn_path, syn_ts)
            print(f"  Generation: {(time.time()-t0)/60:.1f} min")

    else:
        raise ValueError(f"Unknown protocol: {protocol}. Use 'authors', 'split', or 'random_init'.")

    # === Per-proportion LSTM training & evaluation ===
    for prop in proportions:
        if prop in results:
            print(f"  Run {run_idx:2d} | Prop {prop:3d}% | already done — MAPE={results[prop]:.5f} (loaded)")
            continue

        syn_len  = int(MIX_LENGTH * prop / 100)
        real_len = MIX_LENGTH - syn_len
        mix_dataset = build_mix_dataset(real_ts, syn_ts, real_len, syn_len, scaler)

        t0 = time.time()
        model_lstm, train_loss = train_lstm_full_batch(
            mix_dataset, seed=seed_base + run_idx + prop)
        train_time = time.time() - t0
        mape = evaluate_lstm(model_lstm, test_dataset, scaler)
        results[prop] = mape

        print(f"  Run {run_idx:2d} | Prop {prop:3d}% | windows={mix_dataset.shape[0]:5d} | "
              f"train_loss={train_loss:.5f} | MAPE={mape:.5f} | LSTM={train_time:.1f}s")

        sorted_props = sorted(results.keys())
        df_save = pd.DataFrame({
            'proportion_pct': sorted_props,
            'mape': [results[p] for p in sorted_props]
        })
        df_save.to_csv(run_csv, index=False)

        del model_lstm, mix_dataset
        torch.cuda.empty_cache()

    return {p: results[p] for p in proportions if p in results}

print(f"✓ TMTR runner ready (protocol={PROTOCOL!r})")


In [ ]:
# ============================================================================
# === MAIN LOOP — runs all replicates with live plotting + mean tracking ===
# ============================================================================
%matplotlib inline
from IPython.display import display, clear_output

all_results = {}
running_mape = np.full((N_RUNS, len(PROPORTIONS)), np.nan)

# === STEP 1: Pre-load any runs already saved on Drive ===
print(f"Pre-loading existing runs from {RES_DIR}...")
for run_idx in range(N_RUNS):
    run_csv = f'{RES_DIR}/run_{run_idx:02d}.csv'
    if os.path.exists(run_csv):
        df = pd.read_csv(run_csv)
        for j, p in enumerate(PROPORTIONS):
            mask = df['proportion_pct'] == p
            if mask.any():
                running_mape[run_idx, j] = df.loc[mask, 'mape'].values[0]
        all_results[run_idx] = dict(zip(df['proportion_pct'], df['mape']))
n_preloaded = int(sum(~np.all(np.isnan(running_mape), axis=1)))
print(f"  ✓ Found {n_preloaded} existing runs on Drive\n")


# === STEP 2: For 'split' protocol, ensure shared continuous trajectory exists ===
split_traj = None
if PROTOCOL == 'split':
    split_traj = _ensure_split_trajectory(N_RUNS, MIX_LENGTH, seed_base=42)


# === STEP 3: Live plot helper ===
def plot_live(elapsed_min=None):
    """Render plot in AUTHORS' EXACT STYLE (replicates plot_dowmstream_tmtr).
    - x-axis: proportion_pct (matches paper Fig. 6a)
    - CI band: trimmed min/max (authors' summarize_results formula)
    """
    valid = ~np.all(np.isnan(running_mape), axis=1)
    n_done = int(valid.sum())
    if n_done == 0:
        return

    clear_output(wait=True)
    fig, ax = plt.subplots()

    if n_done >= 2:
        n_iters = running_mape.shape[1]
        avg = np.full(n_iters, np.nan); mn = np.full(n_iters, np.nan); mx = np.full(n_iters, np.nan)
        for j in range(n_iters):
            res = running_mape[valid, j]
            res = np.sort(res[~np.isnan(res)])
            if len(res) < 2:
                continue
            pencentile = max(int(np.ceil(len(res) * 0.025)), 1)
            if len(res) > 2 * pencentile:
                avg[j] = np.mean(res[pencentile:-pencentile])
            else:
                avg[j] = np.mean(res)
            mn[j] = res[pencentile] if pencentile < len(res) else res[0]
            mx[j] = res[-pencentile] if pencentile < len(res) else res[-1]
        ax.plot(PROPORTIONS, avg)
        ax.fill_between(PROPORTIONS, mn, mx, alpha=0.2)
        if not np.isnan(avg[0]):
            ax.axhline(y=avg[0], color='gray', linestyle='--')
    else:
        ax.plot(PROPORTIONS, running_mape[0])
        if not np.isnan(running_mape[0, 0]):
            ax.axhline(y=running_mape[0, 0], color='gray', linestyle='--')

    ax.set_xlabel('Syn. Prop. (%)')
    ax.set_ylabel('MAPE')
    title = f'TMTR — {PRETTY_NAME} (K={K}, {PROTOCOL!r}) — {n_done}/{N_RUNS} runs'
    if elapsed_min is not None:
        title += f' (last: {elapsed_min:.1f} min)'
    ax.set_title(title, fontsize=10)
    plt.tight_layout()
    fig.savefig(f'{FIG_DIR}/live.png', dpi=120, bbox_inches='tight')
    display(fig)
    plt.close(fig)


# === STEP 4: Run the loop ===
todo_runs = [r for r in range(N_RUNS) if not (
    np.all(~np.isnan(running_mape[r])) and len(all_results.get(r, {})) >= len(PROPORTIONS)
)]

if not todo_runs:
    print(f"✓ All {N_RUNS} runs already complete — nothing to do.")
else:
    print(f"▶ Running {len(todo_runs)} pending runs ({todo_runs})\n")
    for run_idx in todo_runs:
        t0 = time.time()
        print(f"\n=== Run {run_idx+1}/{N_RUNS} ===")
        run_res = run_single_tmtr(run_idx, PROPORTIONS, seed_base=42,
                                  protocol=PROTOCOL, split_trajectory=split_traj)
        for j, p in enumerate(PROPORTIONS):
            if p in run_res:
                running_mape[run_idx, j] = run_res[p]
        all_results[run_idx] = run_res
        elapsed = (time.time() - t0) / 60
        plot_live(elapsed_min=elapsed)
        print(f"=== Run {run_idx+1}/{N_RUNS} done in {elapsed:.1f} min ===\n")

print(f"\n✓ Loop complete. Per-run CSVs in {RES_DIR}")


## 5. Phase 3 — Aggregation & Confidence Intervals


In [ ]:
# Reload all per-run CSVs from Drive (handles re-runs, partial completion)
rows = []
for run_idx in range(N_RUNS):
    p = f'{RES_DIR}/run_{run_idx:02d}.csv'
    if not os.path.exists(p):
        print(f"⚠ Missing run {run_idx}")
        continue
    df = pd.read_csv(p)
    df['run_idx'] = run_idx
    rows.append(df)

df_all = pd.concat(rows, ignore_index=True)
df_pivot = df_all.pivot(index='run_idx', columns='proportion_pct', values='mape')
print(f"Results matrix: {df_pivot.shape} (runs × proportions)")
print("\nMAPE per run × synthetic proportion (%):")
print(df_pivot.round(5))


In [ ]:
# === Phase 3 — Aggregate using AUTHORS' formula (trimmed min/max) ===
# Replicates experiments/utils_downstream.py::summarize_results.

def summarize_authors_style(errors_matrix):
    n_runs, n_iters = errors_matrix.shape
    summary = np.zeros((3, n_iters))
    for i in range(n_iters):
        res = errors_matrix[:, i].copy()
        res.sort()
        pencentile = max(int(np.ceil(len(res) * 0.025)), 1)
        summary[0, i] = np.mean(res[pencentile:-pencentile]) if len(res) > 2*pencentile else np.mean(res)
        summary[1, i] = res[pencentile]
        summary[2, i] = res[-pencentile]
    return pd.DataFrame({'avg': summary[0, :], 'min': summary[1, :], 'max': summary[2, :]})


errors_matrix = df_pivot[PROPORTIONS].to_numpy()
summary_df_authors = summarize_authors_style(errors_matrix)
summary_df_authors['proportion_pct'] = PROPORTIONS
summary_df_authors = summary_df_authors[['proportion_pct', 'avg', 'min', 'max']]
summary_df_authors['n_runs'] = (~np.isnan(errors_matrix)).sum(axis=0)

print("Summary (authors-style: trimmed mean + min/max band):")
print(summary_df_authors.round(5).to_string(index=False))

summary_df_authors.to_csv(f'{RES_DIR}/summary_authors_style.csv', index=False)
df_pivot.to_csv(f'{RES_DIR}/results_matrix.csv')
print(f"\n✓ Saved to {RES_DIR}/summary_authors_style.csv and results_matrix.csv")

# Bootstrap CI version
def bootstrap_ci(x, n_boot=10000, ci=0.95, seed=0):
    rng = np.random.RandomState(seed)
    n = len(x)
    boots = np.array([rng.choice(x, size=n, replace=True).mean() for _ in range(n_boot)])
    lo, hi = np.quantile(boots, [(1-ci)/2, 1-(1-ci)/2])
    return lo, hi

summary_stats = []
for p in PROPORTIONS:
    vals = df_pivot[p].dropna().values
    if len(vals) == 0:
        continue
    lo, hi = bootstrap_ci(vals)
    summary_stats.append({
        'proportion_pct': p, 'mean_mape': vals.mean(), 'std_mape': vals.std(),
        'ci95_low': lo, 'ci95_high': hi, 'n_runs': len(vals),
    })
summary_df = pd.DataFrame(summary_stats)
summary_df.to_csv(f'{RES_DIR}/summary.csv', index=False)
print(f"✓ Bootstrap-CI summary at {RES_DIR}/summary.csv")


## 6. Final Figures — Replicate Fig. 6(a) of the Paper


In [ ]:
# === Phase 4 — Final figures: BOTH styles (authors + enhanced) ===

# --- 6a. AUTHORS' EXACT STYLE ---
# Replicates experiments/utils_downstream.py::plot_dowmstream_tmtr verbatim.
fig, ax = plt.subplots()

error_avg = summary_df_authors['avg'].values
error_min = summary_df_authors['min'].values
error_max = summary_df_authors['max'].values

ax.plot(PROPORTIONS, error_avg)
ax.fill_between(PROPORTIONS, error_min, error_max, alpha=0.2)
ax.axhline(y=error_avg[0], color='gray', linestyle='--')
ax.set_xlabel('Syn. Prop. (%)')
ax.set_ylabel('MAPE')

plt.tight_layout()
fig_path = f'{FIG_DIR}/final.pdf'
fig.savefig(fig_path, bbox_inches='tight')
fig.savefig(fig_path.replace('.pdf', '.png'), bbox_inches='tight', dpi=150)
plt.show()
plt.close(fig)
print(f"✓ Authors-style figure saved to {fig_path}")


# --- 6b. ENHANCED STYLE (markers + bootstrap CI + value annotations) ---
fig2, ax2 = plt.subplots(figsize=(11, 6))

mean = summary_df['mean_mape'].values
lo   = summary_df['ci95_low'].values
hi   = summary_df['ci95_high'].values
xv   = summary_df['proportion_pct'].values
n_runs_actual = int(summary_df['n_runs'].iloc[0])

ax2.plot(xv, mean, 'o-', linewidth=2.5, color='steelblue', markersize=7,
         label=f'FTS-Diffusion (n={n_runs_actual} runs)')
ax2.fill_between(xv, lo, hi, alpha=0.25, color='steelblue', label='95% bootstrap CI')
ax2.axhline(mean[0], color='gray', linestyle='--', alpha=0.7,
            label=f'Real-only baseline (0%) = {mean[0]:.4f}')

for x, y in zip(xv, mean):
    if np.isnan(y): continue
    ax2.annotate(f'{y:.4f}', xy=(x, y), xytext=(0, 12),
                 textcoords='offset points', ha='center', fontsize=9,
                 fontweight='bold', color='darkblue',
                 bbox=dict(boxstyle='round,pad=0.3', fc='white',
                           ec='darkblue', alpha=0.85, linewidth=0.8))

# % change annotation at 100% synthetic
if PROPORTIONS[-1] == 100 and not np.isnan(mean[-1]) and not np.isnan(mean[0]):
    pct_change = 100 * (mean[-1] - mean[0]) / mean[0]
    ax2.annotate(f'ΔMAPE @ 100%: {pct_change:+.1f}%',
                 xy=(xv[-1], mean[-1]), xytext=(xv[-1]*0.55, mean[-1] + 0.02),
                 arrowprops=dict(arrowstyle='->', color='red'),
                 fontsize=11, color='red')

ax2.set_xlabel('Synthetic Proportion (%)', fontsize=12)
ax2.set_ylabel('MAPE (real test set)', fontsize=12)
ax2.set_title(f'TMTR — {PRETTY_NAME} (K={K}, {PROTOCOL!r}) — '
              f'{n_runs_actual} runs × {len(PROPORTIONS)} proportions',
              fontsize=12, fontweight='bold')
ax2.legend(loc='upper right', fontsize=10)
ax2.grid(alpha=0.3)
ymin, ymax = ax2.get_ylim()
ax2.set_ylim(ymin, ymax + 0.015)
plt.tight_layout()

fig_path_e = f'{FIG_DIR}/final_enhanced.pdf'
fig2.savefig(fig_path_e, bbox_inches='tight', dpi=150)
fig2.savefig(fig_path_e.replace('.pdf', '.png'), bbox_inches='tight', dpi=150)
plt.show()
plt.close(fig2)
print(f"✓ Enhanced-style figure saved to {fig_path_e}")


## 7. Quality Checks


In [ ]:
# === Sanity checks ===
print("=" * 70)
print(f"TMTR sanity report — {PRETTY_NAME} (K={K}, protocol={PROTOCOL!r})")
print("=" * 70)

n_done = (~df_pivot.isna()).all(axis=1).sum()
n_partial = ((~df_pivot.isna()).any(axis=1) & df_pivot.isna().any(axis=1)).sum()
print(f"  Runs fully complete:  {n_done} / {N_RUNS}")
print(f"  Runs partial:         {n_partial}")
print(f"  Proportions per run:  {len(PROPORTIONS)}")
print()

# Direction check: does MAPE increase with synthetic proportion (expected for poor synth)?
if not np.isnan(summary_df['mean_mape'].iloc[0]) and not np.isnan(summary_df['mean_mape'].iloc[-1]):
    delta = summary_df['mean_mape'].iloc[-1] - summary_df['mean_mape'].iloc[0]
    pct = 100 * delta / summary_df['mean_mape'].iloc[0]
    sign = "↑" if delta > 0 else "↓"
    print(f"  MAPE @ 0%   (real only):  {summary_df['mean_mape'].iloc[0]:.5f}")
    print(f"  MAPE @ 100% (synth only): {summary_df['mean_mape'].iloc[-1]:.5f}")
    print(f"  Δ = {sign} {pct:+.1f}%   (positive = synthetic data hurts forecasting)")
    print()

# Min MAPE proportion
min_idx = summary_df['mean_mape'].idxmin()
min_prop = summary_df['proportion_pct'].iloc[min_idx]
print(f"  Best proportion:      {min_prop}%  (MAPE = {summary_df['mean_mape'].iloc[min_idx]:.5f})")
print()

# Pre-flight checks
assert (df_pivot.values > 0).all() or df_pivot.isna().any().any(), \
    "All MAPE values should be > 0 or NaN"
print("✓ All sanity checks passed")


## 8. Cross-protocol comparison

Once you have run multiple protocols (`authors`, `split`, `random_init`) for
the same `(ASSET, K)`, this section overlays their MAPE-vs-proportion curves
on a single figure for visual comparison.


In [ ]:
# === Cross-protocol comparison plot ===
PROTOCOLS_TO_COMPARE = ['authors', 'split', 'random_init']

def load_protocol_summary(protocol):
    """Load summary CSV for a protocol if it exists."""
    path = f'{PERSIST_ROOT}/results/{EXPERIMENT}/{ASSET}/k{K}/{protocol}/summary.csv'
    if not os.path.exists(path):
        return None
    return pd.read_csv(path)


protocol_data = {}
for p in PROTOCOLS_TO_COMPARE:
    df_p = load_protocol_summary(p)
    if df_p is not None and not df_p.empty:
        protocol_data[p] = df_p
        print(f"  ✓ {p}: {len(df_p)} proportions, {int(df_p['n_runs'].iloc[0])} runs")
    else:
        print(f"  — {p}: not yet available (run notebook with PROTOCOL={p!r})")

if len(protocol_data) >= 2:
    colors = {'authors': 'tab:blue', 'split': 'tab:orange', 'random_init': 'tab:green'}
    fig, ax = plt.subplots(figsize=(11, 6))
    for p, df_p in protocol_data.items():
        c = colors.get(p, 'gray')
        ax.plot(df_p['proportion_pct'], df_p['mean_mape'], 'o-',
                color=c, linewidth=2, markersize=6,
                label=f"{p} (n={int(df_p['n_runs'].iloc[0])})")
        ax.fill_between(df_p['proportion_pct'],
                        df_p['ci95_low'], df_p['ci95_high'],
                        color=c, alpha=0.2)

    # Use 'authors' baseline (or first available) as reference horizontal line
    ref = protocol_data.get('authors', list(protocol_data.values())[0])
    ax.axhline(ref['mean_mape'].iloc[0], color='gray', linestyle='--', alpha=0.6,
               label=f"Real-only baseline = {ref['mean_mape'].iloc[0]:.4f}")

    ax.set_xlabel('Synthetic Proportion (%)', fontsize=12)
    ax.set_ylabel('MAPE (real test set)', fontsize=12)
    ax.set_title(f'TMTR cross-protocol comparison — {PRETTY_NAME} (K={K})',
                 fontsize=13, fontweight='bold')
    ax.legend(loc='upper right', fontsize=10)
    ax.grid(alpha=0.3)
    plt.tight_layout()

    cmp_dir = f'{PERSIST_ROOT}/figures/{EXPERIMENT}/{ASSET}/k{K}'
    os.makedirs(cmp_dir, exist_ok=True)
    fig.savefig(f'{cmp_dir}/cross_protocol.pdf', bbox_inches='tight')
    fig.savefig(f'{cmp_dir}/cross_protocol.png', bbox_inches='tight', dpi=150)
    plt.show()
    plt.close(fig)
    print(f"\n✓ Cross-protocol figure saved to {cmp_dir}/cross_protocol.{{pdf,png}}")
else:
    print(f"\n⚠ Need ≥ 2 protocols with results to draw the comparison — currently have {len(protocol_data)}.")


## 9. How to resume / switch asset / protocol

### Resume an interrupted run
Just re-execute Phase 2. The loop:
1. Pre-loads any `run_XX.csv` already on Drive
2. Skips fully completed runs
3. For partial runs, only computes the missing proportions

### Switch asset / protocol
1. Edit cell 6 (Configuration): change `ASSET` and/or `PROTOCOL`
2. Re-run from Phase 1 onward; existing data on Drive is reused automatically

### Tips
- For `split`, the shared continuous trajectory at `SYN_DIR/_global_continuous.npy`
  is generated **once** for the whole `(ASSET, K, protocol=split)` triple.
  Delete it manually if you want a fresh chain.
- For `random_init`, the per-run synthetic series at `run_XX_syn.npy` are NOT
  deterministic across notebook restarts unless you keep the same seed_base.
- The bootstrap CI uses 10 000 resamples — increase if you need tighter CIs.
